# Extractie Spatiotemporele Data – Dataset C (Puurs)

Uitvoeren van objectdetectie en tracking om de ruwe kinematische data (bounding boxes) te genereren.

> **Vereisten:**
> - Video `P_J16.MP4`
> - Roboflow API-key
>
> **Output:** `P_J16.json`

### AI-Inferentie en Traject-Extractie

In [2]:
import cv2
import json
import supervision as sv
from inference import get_model
from tqdm.notebook import tqdm

# --- INSTELLINGEN ---
VIDEO_PATH = '../../data/video/P_J16.MP4'
JSON_OUTPUT = '../../data/detecties/P_J16.json'
ROBOFLOW_API_KEY = "API_KEY"
BEST_MODEL_ID = "tracking-runners-puurs/1"

CONFIDENCE_THRESHOLD = 0.55
LOST_TRACK_BUFFER = 30

# --- INITIALISATIE ---
model = get_model(model_id=BEST_MODEL_ID, api_key=ROBOFLOW_API_KEY)
tracker = sv.ByteTrack(lost_track_buffer=LOST_TRACK_BUFFER)
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

export_data = {
    "fps": video_info.fps,
    "total_frames": video_info.total_frames,
    "tracks": {}
}

print(f"Start extractie van {video_info.total_frames} frames ({video_info.fps} fps)...")

# --- INFERENTIE LOOP ---
for frame_idx in tqdm(range(video_info.total_frames), desc="Posities extraheren"):
    success, frame = cap.read()
    if not success:
        break

    results = model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)
    detections = detections[detections.confidence > CONFIDENCE_THRESHOLD]
    tracks = tracker.update_with_detections(detections=detections)

    if tracks.tracker_id is not None:
        for i, t_id in enumerate(tracks.tracker_id):
            t_id = int(t_id)
            bbox = tracks.xyxy[i]

            x_center = float((bbox[0] + bbox[2]) / 2.0)
            y_center = float((bbox[1] + bbox[3]) / 2.0)
            y_bottom = float(bbox[3])

            if str(t_id) not in export_data["tracks"]:
                export_data["tracks"][str(t_id)] = {
                    "frames": [],
                    "x_coords": [],
                    "y_coords": [],
                    "y_bottoms": []
                }

            export_data["tracks"][str(t_id)]["frames"].append(int(frame_idx))
            export_data["tracks"][str(t_id)]["x_coords"].append(x_center)
            export_data["tracks"][str(t_id)]["y_coords"].append(y_center)
            export_data["tracks"][str(t_id)]["y_bottoms"].append(y_bottom)

cap.release()

# --- JSON OPSLAAN ---
with open(JSON_OUTPUT, 'w') as f:
    json.dump(export_data, f)

print(f"\n✅ Extractie voltooid! Data opgeslagen in: {JSON_OUTPUT}")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


[06/01/26 21:48:52] WARNING  Your inference package version 0.45.1 is out of date! Please upgrade to ]8;id=400840;file://C:\Users\32471\AppData\Local\Programs\Python\Python311\Lib\site-packages\inference\core\__init__.py\__init__.py]8;;\:]8;id=616617;file://C:\Users\32471\AppData\Local\Programs\Python\Python311\Lib\site-packages\inference\core\__init__.py#41\41]8;;\
                             version 1.2.12 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

ModelDependencyMissing: Your `inference` configuration does not support PaliGemma model. Use pip install 'inference[transformers]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does not support Florence2 model. Use pip install 'inference[transformers]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does not support Qwen2.5-VL model. Use pip install 'inference[transformers]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[clip]' to install missing requirements.
ModelDependencyMissing: Your `inference` configuration does

Start extractie van 3026 frames (29.97002997002997 fps)...


Posities extraheren:   0%|          | 0/3026 [00:00<?, ?it/s]

KeyboardInterrupt: 